In [127]:
# 1. Import librerie e utility
import importlib
import os
import sys
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo')
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/general/src')

import matplotlib.pyplot as plt

import ting_utils
importlib.reload(ting_utils)
from ting_utils import (
    collect_curve_files,
    collect_fd_folders,
    process_curve_ting,
    run_ting_analysis,
    save_ting_outputs,
    select_data_folder_gui,
)

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 120, 'font.size': 11})


In [128]:
# 2. Selezione del dataset: finestra grafica oppure percorso manuale
USE_GUI_SELECTOR = False  # metti True per aprire la finestra grafica
ROOT_DATA_DIR = r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo"
MANUAL_FALLBACK_PATH = r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626"   

if USE_GUI_SELECTOR:
    selected_dir = select_data_folder_gui(
        initial_dir=ROOT_DATA_DIR,
        title='Seleziona la cartella della data (es. 150426) oppure una cartella FD',
    )
    if not selected_dir:
        raise RuntimeError('Selezione annullata: nessuna cartella scelta.')
    data_dir = selected_dir
else:
    data_dir = MANUAL_FALLBACK_PATH

print(f"Input selezionato: {data_dir}")


Input selezionato: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626


In [129]:
# 3. Controllo input / preprocessing iniziale
if os.path.isdir(data_dir):
    fd_folders = collect_fd_folders(data_dir)
    curve_files = collect_curve_files(data_dir)

    print(f"Cartella selezionata: {data_dir}")
    if fd_folders:
        print(f"Cartelle FD rilevate automaticamente: {len(fd_folders)}")
        print('Cell trovate:')
        for fd_path in fd_folders[:12]:
            print(f" - {os.path.basename(os.path.dirname(fd_path))}/FD")
        if len(fd_folders) > 12:
            print(f" ... e altre {len(fd_folders) - 12} cartelle FD")
    else:
        print('Nessuna cartella FD dedicata trovata: cerco curve direttamente nella cartella selezionata.')

    print(f"Curve totali trovate: {len(curve_files)}")
    if not curve_files:
        raise FileNotFoundError('Nessuna curva supportata trovata nella cartella selezionata.')

    print('Prime curve rilevate:')
    for path in curve_files[:10]:
        print(f" - {os.path.basename(path)}")
    if len(curve_files) > 10:
        print(f" ... e altre {len(curve_files) - 10}")
else:
    curve_data = process_curve_ting(data_dir)

    FC = curve_data['FC_object']
    param_dict = curve_data['param_dict']
    baseline_info = curve_data['baseline_info']

    print(f"Tipo file: {curve_data['metadata'].get('file_type', 'n/d')}")
    print(f"k = {curve_data['spring_constant']:.4f} N/m")
    print(f"Deflection sensitivity = {curve_data['defl_sens'] * 1e9:.2f} nm/V")
    print(f"Baseline correction status: {baseline_info.get('status')}")


Cartella selezionata: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626
Cartelle FD rilevate automaticamente: 4
Cell trovate:
 - cell01/FD
 - cell02/FD
 - cell03/FD
 - cell04/FD
Curve totali trovate: 40
Prime curve rilevate:
 - Area1-0000.jpk-force
 - Area1-0002.jpk-force
 - Area1-0004.jpk-force
 - Area1-0006.jpk-force
 - Area1-0008.jpk-force
 - Area1-0010.jpk-force
 - Area1-0012.jpk-force
 - Area1-0014.jpk-force
 - Area1-0016.jpk-force
 - Area1-0018.jpk-force
 ... e altre 30


In [ ]:
# 4. Fit Ting
SMOOTH_WIN = 5

# ── Correzioni baseline + opzioni fit E0 ──
BASELINE_CORRECTIONS = {
    "correct_tilt":             True,    # tilt/inclinazione baseline
    "apply_baseline_alignment": False,    # allineamento baseline generico
    "baseline_align_poc":       False,    # allineamento baseline al PoC
    "apply_ramp_correction":    True,    # correzione ramp di chiusura
    "vdragcorr":                True,     # correzione viscous drag (forza idrodinamica)

    # Disattivo il vincolo: E0 Ting puo superare E0 Hertz iniziale
    "enforce_hertz_e0_upper_bound": False,
}

analysis_results = run_ting_analysis(
    data_dir,
    raw_deflection_smooth_win=SMOOTH_WIN,
    param_overrides=BASELINE_CORRECTIONS,
)



Analisi batch: trovate 40 curve in 4 cartelle FD sotto /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626
[1/40] Analisi di Area1-0000 (cell01)


/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161

[2/40] Analisi di Area1-0002 (cell01)


/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161

[3/40] Analisi di Area1-0004 (cell01)


...

[39/40] Analisi di Area4-0002 (cell04)


/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161

[40/40] Analisi di Area4-0004 (cell04)


/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(indentation, force, sample_height)**2/force)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/ting.py:269: RuntimeWarning: divide by zero encountered in divide
  a = (self.get_residuals(time, F, delta, t0, idx_tm, smooth_w, v0t, v0r)**2/F)
/Users/furbetta/miniconda3/envs/afm-env/lib/python3.9/site-packages/pyfmrheo/models/hertz.py:161


Young (Hertz) per cellula:
  cell01: E_mean=1.992e+02 Pa (0.199 kPa) | E_median=1.838e+02 Pa | n=11
  cell02: E_mean=2.646e+02 Pa (0.265 kPa) | E_median=2.424e+02 Pa | n=21
  cell03: E_mean=1.745e+02 Pa (0.174 kPa) | E_median=1.632e+02 Pa | n=5
  cell04: E_mean=2.680e+02 Pa (0.268 kPa) | E_median=2.627e+02 Pa | n=3


In [131]:
# 5. Riepilogo risultati
is_batch = analysis_results.get('is_batch', False)

if is_batch:
    n_found   = analysis_results.get('n_found', 0)
    n_success = analysis_results.get('n_success', 0)
    n_failed  = analysis_results.get('n_failed', 0)
    print(f"Analisi batch: {n_found} curve trovate | {n_success} ok | {n_failed} fallite")

    summary_rows = analysis_results.get('summary_rows', [])
    if summary_rows:
        import pandas as pd
        df = pd.DataFrame(summary_rows)
        _cols = [c for c in [
            'curve', 'cell_folder', 'status', 'selected_model',
            'hertz_E0_init_kpa',
            'young_hertz_kpa',
            'E0_Pa', 'betaE', 'tc_ms', 'r2_plr',
            'r2_gm2', 'rmse_plr_pN',
        ] if c in df.columns]
        display(df[_cols])
else:
    selected_model   = analysis_results.get('selected_model', 'n/d')
    selection_reason = analysis_results.get('selection_reason', '')
    r2_vis           = analysis_results.get('r2_vis', float('nan'))
    r2_gm2           = analysis_results.get('r2_gm2', float('nan'))
    gm2_ok           = analysis_results.get('gm2_ok', False)
    retrace_trim     = analysis_results.get('retrace_trim_points', 0)
    contact_info     = analysis_results.get('contact_point_info', {})
    drag_info        = analysis_results.get('force_drag_info', {})
    closure          = analysis_results.get('closure_refinement', {})
    hertz_result     = analysis_results.get('hertz_result')

    print(f"Modello selezionato : {selected_model}  ({selection_reason})")
    print(f"R² Ting-PLR         : {r2_vis:.4f}")
    print(f"R² Ting-GM2         : {r2_gm2:.4f}  (gm2_ok={gm2_ok})")
    if hertz_result is not None and hasattr(hertz_result, 'E0'):
        print(f"Hertz E0 init       : {float(hertz_result.E0):.3e} Pa ({float(hertz_result.E0)/1e3:.3f} kPa)")
    print(f"Punti retrace tagliati: {retrace_trim}")
    print(f"Contact strategy    : {contact_info.get('selected_strategy', 'n/d')}")
    print(f"Drag correction     : applied={drag_info.get('applied', False)}  Δ={drag_info.get('correction_N', 0)*1e12:.2f} pN")
    print(f"Closure refinement  : applied={closure.get('applied', False)}")


Analisi batch: 40 curve trovate | 40 ok | 0 fallite


,curve,cell_folder,status,selected_model,hertz_E0_init_kpa,young_hertz_kpa,E0_Pa,betaE,tc_ms,r2_plr,r2_gm2,rmse_plr_pN
0,Area1-0000,cell01,ok,PLR,0.144595,0.144595,26.579042,0.395857,2.100000e+01,0.556934,None,137.083126
1,Area1-0002,cell01,ok,PLR,0.111570,0.111570,1.849163,0.440032,4.569066e-10,0.738614,None,114.270183
2,Area1-0004,cell01,ok,PLR,0.163394,0.163394,108.951329,0.149744,2.399997e+01,0.887876,None,92.146526
3,Area1-0006,cell01,ok,PLR,0.320156,0.320156,3.316428,0.436595,3.940266e-09,0.836086,None,132.413423
4,Area1-0008,cell01,ok,PLR,0.218474,0.218474,49.707511,0.383934,2.200000e+01,0.910057,None,96.076372
5,Area1-0010,cell01,ok,PLR,0.154837,0.154837,37.988522,0.393669,2.500000e+01,0.891725,None,121.281014
6,Area1-0012,cell01,ok,PLR,0.216124,0.216124,47.753155,0.391431,2.200000e+01,0.914060,None,97.083269
7,Area1-0014,cell01,ok,PLR,0.064364,0.064364,1.659751,0.441865,1.104385e+01,0.788914,None,89.112763
8,Area1-0016,cell01,ok,PLR,0.331649,0.331649,75.265128,0.377116,1.900000e+01,0.924595,None,88.542106
9,Area1-0018,cell01,ok,Maxwell,0.183788,0.183788,1.988193,0.344811,1.055231e+02,0.210158,None,209.308667


In [132]:
# 6. Salvataggio
save_ting_outputs(
    data_dir,
    analysis_results,
    open_output_folder=False,
    save_each_curve=True,
    show_plots=True,
    save_json=False,
    save_csv=True,
    save_summary_table=True,
)


Cartella batch di salvataggio: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626
Tabella batch salvata: ting_batch_summary_table.pdf
CSV batch salvato: ting_batch_summary.csv
Cartella di salvataggio: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD
CSV salvato: Area1-0000_ting_fit_data.csv
PNG salvato: Area1-0000_baseline_comparison.png
PNG salvato: Area1-0000_ting_force_vs_ind.png
PNG salvato: Area1-0000_ting_force_vs_time.png
Cartella di salvataggio: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD
CSV salvato: Area1-0002_ting_fit_data.csv
PNG salvato: Area1-0002_baseline_comparison.png
PNG salvato: Area1-0002_ting_force_vs_ind.png
PNG salvato: Area1-0002_ting_force_vs_time.png
Cartella di salvataggio: /Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD
CSV salvato: Area1-0004_ting_fit_data.csv
PNG salvato: Area1-0004_baseline_comparison.png
PNG salvato: Area1-0004_ti

{'save_dir': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626',
 'summary_table_path': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/ting_batch_summary_table.pdf',
 'summary_json_path': None,
 'summary_csv_path': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/ting_batch_summary.csv',
 'n_saved_curves': 40,
 'individual_outputs': [{'save_dir': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD',
   'curve_name': 'Area1-0000',
   'json_path': None,
   'csv_path': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD/Area1-0000_ting_fit_data.csv',
   'plot_baseline_comparison': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD/Area1-0000_baseline_comparison.png',
   'plot_force_vs_indentation': '/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626/cell01/FD/Area1-0000_ting_force_vs_ind.png',
   'plot_f